## tl;dr

- 当前数据不支持经典题目级 IRT；本 notebook 把 benchmark family 当作连续响应 item。
- 建议先把每板块至少 2 个测试家族、3 个为软目标作为影子榜规则；当前不直接替换生产榜。
- 测量层与产品规则分开：Fable 5 固定第一，Qwen 遵守显式版本/tier 偏序，Gemini Flash 前至少有 15 个 Main-evidence 开源模型。
- 五套约束方案都输出完整前 50；首选约束 1PL/Rasch 做影子榜，2PL 只做 challenger。

## Context & Methods

分析控制源为 `docs/data/models.json`。模型档位按网站现行 `variantPriority` 规则去重；同板块内的 HLE、GPQA、AIME 等同义来源合并为一个测试家族。百分比响应进入 empirical-logit，Elo 先转为 benchmark 内百分位。

### Key Assumptions

- 缺失明显不是随机缺失，因此后验收缩和保守下界只能缓解，不能消除来源选择偏差。
- 覆盖少于 8 个模型家族的 benchmark 不参与拟合；少于 50 个观测时，2PL discrimination 固定为 1。
- 五个板块各至少 2 个测试家族才进入替代主榜；只有 2 个时仍按不确定性扣分。
- Main evidence 还要求每板块至少 3 项且至少 9 个唯一 benchmark family；未达到者标为 Provisional。
- Fable、Qwen 和 Gemini Flash 是显式产品约束，不解释为 IRT 自然推导的能力结论。

In [1]:
from pathlib import Path
import csv
import json
import sys

sys.dont_write_bytecode = True
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'docs' / 'data' / 'models.json').exists():
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next(path for path in candidates if (path / 'docs' / 'data' / 'models.json').exists())
ANALYSIS_DIR = PROJECT_ROOT / 'analysis' / 'irt_leaderboard_exploration'
sys.path.insert(0, str(ANALYSIS_DIR))
from irt_leaderboard_analysis import run_analysis

result = run_analysis(write_outputs=False, run_stability=False)
summary = result['summary']
print({
    'data_generated_at': summary['data_generated_at'],
    'source_model_rows': summary['source_model_rows'],
    'deduped_scorable_models': summary['deduped_scorable_models'],
    'hard_floor_eligible_models': summary['hard_floor_eligible_models'],
    'all_boards_ge_soft_target_models': summary['all_boards_ge_soft_target_models'],
    'unique_eligible_benchmark_families': summary['unique_eligible_benchmark_families'],
})

{'data_generated_at': '2026-08-06T13:48:16+00:00', 'source_model_rows': 581, 'deduped_scorable_models': 410, 'hard_floor_eligible_models': 328, 'all_boards_ge_soft_target_models': 129, 'unique_eligible_benchmark_families': 26}


## Data

下面检查每个板块在 canonical benchmark family 口径下达到 2 项和 3 项的覆盖率。

In [2]:
for row in result['coverage_profile']:
    print(
        f"{row['board']:<24} items={row['canonical_coverage_items']:>2} "
        f">=2: {row['models_ge_2']:>3}/{row['models']} ({row['share_ge_2']:.1%}) "
        f">=3: {row['models_ge_3']:>3}/{row['models']} ({row['share_ge_3']:.1%})"
    )

Coding                   items= 7 >=2: 400/410 (97.6%) >=3: 269/410 (65.6%)
Agentic/tool work        items=10 >=2: 328/410 (80.0%) >=3: 137/410 (33.4%)
Hard reasoning           items= 6 >=2: 404/410 (98.5%) >=3: 342/410 (83.4%)
Knowledge/science        items= 6 >=2: 406/410 (99.0%) >=3: 401/410 (97.8%)
Instruction/context      items= 5 >=2: 339/410 (82.7%) >=3: 302/410 (73.7%)


## Results

先看五套方案的前 10。分数只在同一方案内解释，跨方案应比较名次而不是绝对分数。

In [3]:
top50 = result['top50']
for scheme_id in ['baseline_aindex', 'rasch_business', 'twopl_equal', 'robust_eb', 'borda_breadth']:
    rows = [row for row in top50 if row['scheme_id'] == scheme_id and row['rank'] <= 10]
    print('\n' + rows[0]['scheme'])
    for row in rows:
        print(f"{row['rank']:>2}. {row['model']} ({row['creator']}) score={row['score']:.2f}")


现行 AIndex（基准）
 1. Claude Fable 5 (with fallback) (Anthropic) score=56.78
 2. GPT-5.6 Sol (max) (OpenAI) score=53.82
 3. Claude Opus 5 (max) (Anthropic) score=52.54
 4. GPT-5.5 (xhigh) (OpenAI) score=51.89
 5. Kimi K3 (max) (Kimi) score=51.23
 6. Claude Opus 4.8 (max) (Anthropic) score=51.12
 7. GPT-5.6 Terra (max) (OpenAI) score=50.19
 8. Claude Opus 4.7 (max) (Anthropic) score=49.49
 9. GPT-5.4 (xhigh) (OpenAI) score=48.59
10. Gemini 3.1 Pro Preview (Google) score=47.97

连续 1PL/Rasch 近似 + 现行板块权重
 1. Claude Fable 5 (with fallback) (Anthropic) score=97.38
 2. GPT-5.6 Sol (max) (OpenAI) score=97.06
 3. GPT-5.6 Terra (max) (OpenAI) score=95.77
 4. GPT-5.5 (xhigh) (OpenAI) score=95.38
 5. Claude Opus 4.8 (max) (Anthropic) score=95.17
 6. GPT-5.4 (xhigh) (OpenAI) score=94.19
 7. Claude Opus 4.7 (max) (Anthropic) score=93.52
 8. Gemini 3.1 Pro Preview (Google) score=93.00
 9. GLM-5.2 (max) (Z AI) score=92.76
10. Qwen3.7 Max (Alibaba) score=92.17

强收缩连续 2PL 近似 + 五板块等权
 1. GPT-5.6 Sol (max) (

In [4]:
diagnostics_path = ANALYSIS_DIR / 'outputs' / 'scheme_diagnostics.csv'
with diagnostics_path.open(encoding='utf-8-sig', newline='') as handle:
    diagnostics = list(csv.DictReader(handle))

for row in diagnostics:
    print({
        'scheme': row['scheme'],
        'ranked_models': int(row['ranked_models']),
        'top50_overlap_with_baseline': int(row['top50_overlap_with_baseline']),
        'spearman_vs_baseline': float(row['spearman_vs_baseline']),
        'lobo_mean_top50_retention': row['lobo_mean_top50_retention'] or None,
        'lobo_min_top50_retention': row['lobo_min_top50_retention'] or None,
        'lobo_min_eligible_population_retention': row['lobo_min_eligible_population_retention'] or None,
    })

{'scheme': '现行 AIndex（基准）', 'ranked_models': 410, 'top50_overlap_with_baseline': 50, 'spearman_vs_baseline': 1.0, 'lobo_mean_top50_retention': None, 'lobo_min_top50_retention': None, 'lobo_min_eligible_population_retention': None}
{'scheme': '连续 1PL/Rasch 近似 + 现行板块权重', 'ranked_models': 328, 'top50_overlap_with_baseline': 41, 'spearman_vs_baseline': 0.985808, 'lobo_mean_top50_retention': '0.941538', 'lobo_min_top50_retention': '0.76', 'lobo_min_eligible_population_retention': '0.396341'}
{'scheme': '强收缩连续 2PL 近似 + 五板块等权', 'ranked_models': 328, 'top50_overlap_with_baseline': 45, 'spearman_vs_baseline': 0.985397, 'lobo_mean_top50_retention': '0.95', 'lobo_min_top50_retention': '0.7', 'lobo_min_eligible_population_retention': '0.396341'}
{'scheme': '稳健秩变换 + 贝叶斯式收缩', 'ranked_models': 328, 'top50_overlap_with_baseline': 39, 'spearman_vs_baseline': 0.978614, 'lobo_mean_top50_retention': '0.943077', 'lobo_min_top50_retention': '0.6', 'lobo_min_eligible_population_retention': '0.396341'}
{'sche

### Product constraint layer

下面把覆盖校正与产品偏序叠加到五套测量方案上。验收重点是 Fable 榜首、Qwen 七条直接边、Gemini Flash 开源下限和约束位移。

In [5]:
from constrained_ranking_analysis import build_constrained_rankings

constrained = build_constrained_rankings(result)
for row in constrained['diagnostics']:
    print({
        'scheme': row['scheme'],
        'fable_rank': row['fable_rank'],
        'qwen_violations_after': row['qwen_direct_edge_violations_after'],
        'flash_main_open_above': row['gemini_flash_min_main_open_models_above'],
        'top50_provisional': row['top50_provisional_models'],
        'max_constraint_shift': row['max_absolute_constraint_shift'],
    })

assert constrained['summary']['validation']['all_schemes_fable_rank_one']
assert constrained['summary']['validation']['all_schemes_qwen_constraints_satisfied']
assert constrained['summary']['validation']['all_schemes_gemini_flash_open_floor_satisfied']
assert constrained['summary']['validation']['each_scheme_has_50_rows']

{'scheme': '覆盖校正 AIndex + 产品硬约束', 'fable_rank': 1, 'qwen_violations_after': 0, 'flash_main_open_above': 15, 'top50_provisional': 13, 'max_constraint_shift': 42}
{'scheme': '连续 1PL/Rasch + 独立覆盖校正 + 产品硬约束', 'fable_rank': 1, 'qwen_violations_after': 0, 'flash_main_open_above': 15, 'top50_provisional': 12, 'max_constraint_shift': 72}
{'scheme': '强收缩连续 2PL + 独立覆盖校正 + 产品硬约束', 'fable_rank': 1, 'qwen_violations_after': 0, 'flash_main_open_above': 15, 'top50_provisional': 12, 'max_constraint_shift': 63}
{'scheme': '稳健秩收缩 + 独立覆盖校正 + 产品硬约束', 'fable_rank': 1, 'qwen_violations_after': 0, 'flash_main_open_above': 15, 'top50_provisional': 14, 'max_constraint_shift': 53}
{'scheme': '收缩 Borda 广度榜 + 独立覆盖校正 + 产品硬约束', 'fable_rank': 1, 'qwen_violations_after': 0, 'flash_main_open_above': 15, 'top50_provisional': 14, 'max_constraint_shift': 62}


## Takeaways

1. **当前不直接替换生产榜。** 唯一 benchmark family 留一后，最差情况下合格人口只保留约 40%，说明门槛依赖少数关键测试。
2. **约束 1PL/Rasch 作为首要影子榜。** 它保留当前业务板块优先级，并把测量分、覆盖惩罚和产品重排分开记录。
3. **2PL 只作为 challenger。** 只有观测数达到 50 的 item 才自由估 discrimination；其余固定为 1。稳健秩收缩与 Borda 用作敏感性参照。
4. **Main / Provisional 必须公开。** 产品约束可提升证据较少的新型号，但约束位移超过 10 名时需要人工复核。
5. **先修证据治理。** 外部结果存在重复冲突、跨板块复用与缺失非随机；新数据源在接口、时效、许可和模型映射均过关前不进入每日 Action。